<a href="https://colab.research.google.com/github/Ankitp2002/RAG_with_milvus/blob/main/RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# demo not for use
from google.colab import userdata

In [7]:
# # Wrap the brackets in quotes so Colab's bash passes them accurately
# !pip install 'markitdown[pdf]' 'markitdown[all]'

# # Directly force-install the underlying engines used by MarkItDown for PDFs
# !pip install pdfplumber pdfminer.six pymupdf

# # Install the core native SDKs and the local Milvus Lite engine
# !pip install -U pymilvus milvus-lite markitdown groq sentence-transformers

# # Install essential Linux binaries for parsing heavy formats and layout assets in Colab
# !apt-get update && apt-get install -y poppler-utils tesseract-ocr

In [8]:
# import os
# from markitdown import MarkItDown
# from groq import Groq
# from pymilvus import MilvusClient, DataType
# from sentence_transformers import SentenceTransformer

# # =====================================================================
# # 1. SETUP & COLAB LOCAL STORAGE ENVIRONMENT
# # =====================================================================

# # Model configurations
# VISION_MODEL = "meta-llama/llama-4-scout-17b-16e-instruct"
# EMBED_MODEL_NAME = "BAAI/bge-large-en-v1.5"
# DIMENSION = 1024  # Standard dense array dimension output for BGE-Large
# COLLECTION_NAME = "colab_local_rag_collection"

# # Absolute local file path inside the Google Colab environment space
# DB_FILE_PATH = os.path.abspath("/content/local_milvus_storage.db")

# # Ensure Groq API Key is present in the Colab session environment variables
# if not userdata.get('GROQ_API_KEY'):
#     # If not set in Colab secrets, you can manually set it here:
#     # os.environ["GROQ_API_KEY"] = "gsk_..."
#     print("⚠️ WARNING: Please ensure GROQ_API_KEY is configured in your environment or Colab secrets.")

# # Initialize native infrastructure clients
# groq_client = Groq(api_key=userdata.get('GROQ_API_KEY'))
# embedding_model = SentenceTransformer(EMBED_MODEL_NAME)

# # Passing a local file string automatically instantiates Milvus Lite locally
# milvus_client = MilvusClient(uri=DB_FILE_PATH)

# # Build the structural database schema natively
# if not milvus_client.has_collection(COLLECTION_NAME):
#     schema = milvus_client.create_schema(auto_id=True, enable_dynamic_field=True)
#     schema.add_field(field_name="id", datatype=DataType.INT64, is_primary=True)
#     schema.add_field(field_name="vector", datatype=DataType.FLOAT_VECTOR, dim=DIMENSION)
#     schema.add_field(field_name="text", datatype=DataType.VARCHAR, max_length=65535)

#     index_params = milvus_client.prepare_index_params()
#     index_params.add_index(field_name="vector", metric_type="COSINE", index_type="AUTOINDEX")

#     milvus_client.create_collection(
#         collection_name=COLLECTION_NAME,
#         schema=schema,
#         index_params=index_params
#     )
#     print(f"[✓] Milvus Lite database file successfully initialized at: {DB_FILE_PATH}")

# # Bind MarkItDown natively to utilize Llama-4-Scout for visual components
# md_parser = MarkItDown(llm_client=groq_client, llm_model=VISION_MODEL)

# # =====================================================================
# # 2. CORE INSERTION PIPELINE (MANUAL INGEST)
# # =====================================================================

# def ingest_unstructured_document(file_path: str):
#     """
#     Parses document structures locally (tables convert to markdown).
#     Images are securely translated via Llama-4-Scout over Groq.
#     Generates local vector arrays and deposits them straight into the local .db file.
#     """
#     if not os.path.exists(file_path):
#         raise FileNotFoundError(f"Provided path does not exist in Colab workspace: {file_path}")

#     print(f"[*] Parsing file via MarkItDown: {file_path}")
#     conversion = md_parser.convert(file_path)
#     markdown_content = conversion.text_content

#     # Split text cleanly by logical markdown segment boundaries
#     text_chunks = [chunk.strip() for chunk in markdown_content.split("\n\n") if chunk.strip()]

#     payload = []
#     print(f"[*] Transforming {len(text_chunks)} segments into vectors using {EMBED_MODEL_NAME}...")
#     for chunk in text_chunks:
#         # Local transformation calculations
#         vector_embedding = embedding_model.encode(chunk).tolist()
#         payload.append({
#             "vector": vector_embedding,
#             "text": chunk
#         })

#     if payload:
#         milvus_client.insert(collection_name=COLLECTION_NAME, data=payload)
#         print(f"[✓] Local Database Sync Complete: {len(payload)} chunks written to {DB_FILE_PATH}")

In [ ]:

# =====================================================================
# 3. CORE EXTRACTION PIPELINE (DIRECT QUERY)
# =====================================================================

# def search_local_index(query_string: str, top_k: int = 3):
#     """
#     Vectorizes the search question locally via BGE and executes a vector
#     similarity search directly inside your local Milvus SQLite file container.
#     """
#     milvus_client.load_collection(collection_name=COLLECTION_NAME)
#     # Math vector space translation
#     query_vector = embedding_model.encode(query_string).tolist()

#     # Run exact search point query
#     hits = milvus_client.search(
#         collection_name=COLLECTION_NAME,
#         data=[query_vector],
#         limit=top_k,
#         output_fields=["text"],
#         search_params={"metric_type": "COSINE"}
#     )

#     print(f"\n🎯 Top {top_k} Results Found for: '{query_string}'")
#     print("=" * 70)
#     for index, hit in enumerate(hits[0]):
#         print(f"\n[Match {index + 1}] Similarity Score: {hit['distance']:.4f}")
#         print(f"Content Extract:\n{hit['entity']['text']}")
#         print("-" * 50)

In [9]:
! pip install llama-index-vector-stores-milvus llama-index-readers-markitdown llama_index groq llama-index-embeddings-fastembed fastembed

In [13]:
import os
from markitdown import MarkItDown
from groq import Groq
from pymilvus import DataType
from sentence_transformers import SentenceTransformer

# LlamaIndex Native Imports
from llama_index.core import StorageContext, VectorStoreIndex, Settings
from llama_index.embeddings.fastembed import FastEmbedEmbedding
from llama_index.vector_stores.milvus import MilvusVectorStore
from llama_index.readers.markitdown import MarkItDownReader

# =====================================================================
# 1. SETUP & SYSTEM-WIDE CONFIGURATION
# =====================================================================

VISION_MODEL = "meta-llama/llama-4-scout-17b-16e-instruct"
EMBED_MODEL_NAME = "BAAI/bge-large-en-v1.5"
DIMENSION = 1024
COLLECTION_NAME = "colab_local_rag_collection"
DB_FILE_PATH = os.path.abspath("/content/local_milvus_storage.db")

# Global LLM Client for MarkItDown's vision capability
groq_client = Groq(api_key=userdata.get('GROQ_API_KEY'))

# 1. Register embedding model globally into LlamaIndex Settings
Settings.embed_model = FastEmbedEmbedding(model_name=EMBED_MODEL_NAME)

# 2. Setup local Milvus Vector Store via LlamaIndex wrapper
vector_store = MilvusVectorStore(
    uri=DB_FILE_PATH,
    collection_name=COLLECTION_NAME,
    dim=DIMENSION,
    similarity_metric="COSINE",
    overwrite=False  # Keeps existing data intact on reruns
)

# 3. Create the Storage Context pipeline container
storage_context = StorageContext.from_defaults(vector_store=vector_store)

# 4. Bind MarkItDown natively to LlamaIndex Reader
md_parser = MarkItDown(llm_client=groq_client, llm_model=VISION_MODEL)
markitdown_reader = MarkItDownReader()
markitdown_reader._markitdown = md_parser


# =====================================================================
# 2. HIGH-PERFORMANCE INGESTION PIPELINE
# =====================================================================

def ingest_unstructured_document_v2(file_path: str):
    """
    Parses complex file structures via MarkItDown, automatically generates
    embeddings, handles text-splitting safely, and streams directly into
    Milvus Lite without manual loop boilerplates.
    """
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"Provided path does not exist in Colab workspace: {file_path}")

    print(f"[*] Loading and parsing document via LlamaIndex MarkItDown wrapper...")
    # Read file directly into an internal LlamaIndex 'Document' object
    documents = markitdown_reader.load_data(file_path=file_path)

    print(f"[*] Auto-indexing, embedding, and streaming vectors into Milvus Lite...")
    # This single engine handles chunking, embedding, database structure setup, and insertion
    index = VectorStoreIndex.from_documents(
        documents,
        storage_context=storage_context,
        show_progress=True
    )
    print(f"[✓] Synced into {DB_FILE_PATH} successfully via LlamaIndex pipeline.")
    return index

In [14]:
# =====================================================================
# 4. EXECUTION SAMPLE RUNTIME
# =====================================================================
# Upload any unstructured asset (PDF, XLSX, DOCX) to your Colab side panel files
target_sample_doc = os.path.abspath("/content/020161622x.pdf" )
#  Execute processing sequence
# ingest_unstructured_document(target_sample_doc)
ingest_unstructured_document_v2(target_sample_doc)

[*] Loading and parsing document via LlamaIndex MarkItDown wrapper...
[*] Auto-indexing, embedding, and streaming vectors into Milvus Lite...


Applying transformations:   0%|          | 0/1 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/44 [00:00<?, ?it/s]

[✓] Synced into /content/local_milvus_storage.db successfully via LlamaIndex pipeline.


In [15]:
from llama_index.core import VectorStoreIndex
from llama_index.core.schema import NodeWithScore
from typing import List

def search_local_index_v2(query_string: str, top_k: int = 3) -> List[NodeWithScore]:
    """
    Vectorizes the query automatically using the globally registered embed_model,
    queries the local Milvus Lite storage engine, and extracts matched text and scores.
    """
    print(f"[*] Connecting to local Milvus index: '{COLLECTION_NAME}'...")

    # 1. Connect to your existing collection directly via the LlamaIndex storage engine
    # (Uses the `vector_store` object we defined earlier)
    index = VectorStoreIndex.from_vector_store(vector_store=vector_store)

    # 2. Convert the index into a high-performance search retriever engine
    retriever = index.as_retriever(similarity_top_k=top_k)

    print(f"[*] Running vector similarity search for: '{query_string}'")
    # 3. Execute search: LlamaIndex automatically calls BGE to extract query embeddings
    # and computes the cosine similarities natively.
    retrieved_nodes = retriever.retrieve(query_string)

    print(f"\n🎯 Top {len(retrieved_nodes)} Results Found for: '{query_string}'")
    print("=" * 70)

    # 4. Enumerate over LlamaIndex's native 'NodeWithScore' payload
    for idx, node_with_score in enumerate(retrieved_nodes):
        score = node_with_score.score          # Extract distance score
        text_content = node_with_score.text    # Extract text content extract

        print(f"\n[Match {idx + 1}] Similarity Score: {score:.4f}")
        print(f"Content Extract:\n{text_content}")
        print("-" * 50)

    return retrieved_nodes

In [16]:
# search_local_index("Extract structural details or metrics from data charts/tables.")
search_local_index_v2("Extract structural details or metrics from data charts/tables.")

[*] Connecting to local Milvus index: 'colab_local_rag_collection'...
[*] Running vector similarity search for: 'Extract structural details or metrics from data charts/tables.'

🎯 Top 3 Results Found for: 'Extract structural details or metrics from data charts/tables.'

[Match 1] Similarity Score: 0.4254
Content Extract:

--------------------------------------------------

[Match 2] Similarity Score: 0.4352
Content Extract:

--------------------------------------------------

[Match 3] Similarity Score: 0.4443
Content Extract:

--------------------------------------------------


[NodeWithScore(node=TextNode(id_='ef098c9a-e0da-44c1-be97-d54ee50c35cb', embedding=None, metadata={}, excluded_embed_metadata_keys=[], excluded_llm_metadata_keys=[], relationships={}, metadata_template='{key}: {value}', metadata_separator='\n', text='', mimetype='text/plain', start_char_idx=None, end_char_idx=None, text_template='{metadata_str}\n\n{content}'), score=0.425362765789032),
 NodeWithScore(node=TextNode(id_='26a01e5d-b118-4ad4-9e68-0f046c12b13b', embedding=None, metadata={}, excluded_embed_metadata_keys=[], excluded_llm_metadata_keys=[], relationships={}, metadata_template='{key}: {value}', metadata_separator='\n', text='', mimetype='text/plain', start_char_idx=None, end_char_idx=None, text_template='{metadata_str}\n\n{content}'), score=0.43518972396850586),
 NodeWithScore(node=TextNode(id_='400e74cd-bad4-491c-98c4-846594b9614b', embedding=None, metadata={}, excluded_embed_metadata_keys=[], excluded_llm_metadata_keys=[], relationships={}, metadata_template='{key}: {value}', 

In [22]:
import random

def extract_random_data_from_milvus(top_n: int = 4):
    """
    Connects directly to the underlying Milvus Lite engine and grabs
    a random sampling of records out of the existing collection fields.
    """
    # 1. Access the database collection using the client reference
    client = vector_store.client

    if not client.has_collection(COLLECTION_NAME):
        print(f"❌ Collection '{COLLECTION_NAME}' does not exist.")
        return []

    print(f"[*] Fetching records from local storage collection: '{COLLECTION_NAME}'...")

    # 2. Query the database for IDs and Text content strings
    # (Leaving filter empty "" pulls all available rows)
    all_records = client.query(
        collection_name=COLLECTION_NAME,
        filter='text like "%book%"',
        output_fields=["id", "text"]
    )

    total_count = len(all_records)
    print(f"[✓] Successfully read {total_count} total records from file.")

    if total_count == 0:
        print("⚠️ Database is currently empty! Nothing to sample.")
        return []

    # 3. Defensive check: if database has fewer records than requested sampling size
    sample_size = min(top_n, total_count)

    # 4. Pull a non-repeating random subset array
    random_samples = random.sample(all_records, sample_size)

    print(f"\n🎲 Displaying {sample_size} Random Data Extractions:")
    print("=" * 70)

    for idx, record in enumerate(random_samples):
        record_id = record.get("id")
        record_text = record.get("text", "No text field found")

        print(f"\n[Random Sample {idx + 1}] Document Database Row ID: {record_id}")
        print(f"Extracted Content:\n{record_text}")
        print("-" * 50)

    return random_samples

# Execute the sampling call
random_data = extract_random_data_from_milvus(top_n=4)

[*] Fetching records from local storage collection: 'colab_local_rag_collection'...
[✓] Successfully read 22 total records from file.

🎲 Displaying 4 Random Data Extractions:

[Random Sample 1] Document Database Row ID: 284d2bb3-0510-4491-ab75-67f34c404793
Extracted Content:
oraconsultantworkingwithmanyclientsatonce.Thisbookwillhelpyou,asanindividual,todobetterwork.Thisbookisn’ttheoretical—weconcentrateonpracticaltopics,onusingyourexperi-encetomakemoreinformeddecisions.ThewordpragmaticcomesfromtheLatinpragmaticus—“skilledinbusiness”—whichitselfisderivedfromtheGreek,meaning“todo.”Thisisabookaboutdoing.Programmingisacraft.Atitssimplest,itcomesdowntogettingacomputertodowhatyouwantittodo(orwhatyouruserwantsittodo).Asaprogrammer,youarepartlistener,partadvisor,partinterpreter,andpartdictator.Youtrytocaptureelusiverequirementsandﬁndawayofexpressingthemsothatameremachinecandothemjustice.Youtrytodocumentyourworksothatotherscanunderstandit,andyoutrytoengineeryourworksothatotherscanbuildonit.What